# Notebook 2 — Resume NER with fine-tuned DistilBERT (token classification)

**Project:** GenAI Resume Analyser
**Input:** raw resume text → **Output:** spans tagged as Skills, Companies, Degree, Designation, College Name, etc.

## Syllabus topics this notebook covers
1. **Transformers & fine-tuning** — DistilBERT fine-tuned for *token* classification (NER)
2. **NLP preprocessing** — subword tokenization, BIO tagging, label alignment across subtokens
3. **Evaluation** — precision / recall / F1 for sequence labeling with seqeval

## Dataset: Dataturks "Resume Entities for NER"
220 real resumes (indeed.com) manually annotated with character-offset spans. We use the official
[Kaggle mirror](https://www.kaggle.com/datasets/dataturks/resume-entities-for-ner).
**Honest caveat:** 220 resumes is *small* for NER. Expect **entity F1 ≈ 0.60–0.85**, not 0.95.
That is a normal result at this scale and makes a great examiner discussion point (data-hungry transformers).

## How to run
1. Colab: `Runtime` → `Change runtime type` → **T4 GPU** → Save — **do this FIRST, on every fresh upload** (uploads reset the runtime to CPU, and training refuses to start without a GPU)
2. `Runtime` → `Run all` (Kaggle secrets in Colab 🔑, or manual upload when prompted)
3. Training: **~5–10 min** on T4 — **25 epochs** (this tiny dataset needs many short epochs; 5 was too few and provably undertrained the model)
4. Move the zip into the project's `notebooks/artifacts/` folder

## What good results look like
- **Early epochs score LOW (0.0–0.3) — that is NORMAL here**; F1 climbs steadily each epoch and only the BEST epoch's checkpoint is kept
- Final val F1 ≈ 0.60–0.85 (small dataset; per-epoch numbers bounce around)
- Skills / Companies / College are easiest; Name / Years of Experience are the weakest (few examples)
- If the FINAL F1 is still under 0.40 → copy the per-epoch `eval_f1` list and send it to me (the eval cell refuses to save a bad model, so nothing broken ships silently)

In [ ]:
# Install pinned libs; torch stays at Colab's preinstalled CUDA-compatible version; seqeval does sequence-labeling metrics
!pip -q install "transformers>=4.44,<5" "datasets>=2.20,<4" "accelerate>=0.34" "seqeval>=1.2" "kagglehub>=1.0.3" matplotlib seaborn

In [ ]:
# Imports + global seed for reproducible training
import os, re, json, glob, shutil, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
SEED = 42
random.seed(SEED); np.random.seed(SEED)

## 1. Get the dataset
**Kaggle way:** add `KAGGLE_USERNAME` + `KAGGLE_KEY` to Colab secrets (🔑 sidebar icon), values from your `kaggle.json`.
**Manual way:** download [resume-entities-for-ner](https://www.kaggle.com/datasets/dataturks/resume-entities-for-ner) manually and pick the JSON when the upload dialog opens.

In [ ]:
# Download the official Dataturks Kaggle mirror via kagglehub; fall back to a manual upload dialog on any failure
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Kaggle credentials loaded from Colab secrets")
    import kagglehub
    base = kagglehub.dataset_download("dataturks/resume-entities-for-ner")
    DATA_PATH = (glob.glob(os.path.join(base, "**", "*.json*"), recursive=True) or [None])[0]
    assert DATA_PATH, "Downloaded dataset contained no JSON file"
except Exception as e:
    print(f"Kaggle download unavailable ({e}) -> manual upload mode")
    from google.colab import files
    up = files.upload()
    DATA_PATH = list(up.keys())[0]
print("Using dataset file:", DATA_PATH)

In [ ]:
# Parse the annotation file: it is JSONL (one resume per line); fall back to one big JSON array if that fails
raw = open(DATA_PATH, encoding="utf-8-sig").read().strip()
try:
    records = [json.loads(line) for line in raw.splitlines() if line.strip()]
    assert records and "content" in records[0]
except Exception:
    records = json.loads(raw)
    if isinstance(records, dict):
        records = records.get("data", [records])
print(f"Parsed {len(records)} annotated resumes")
print("Keys of a record:", list(records[0].keys()))
ex = records[0]
print("First entity example:", ex["annotation"][0]["label"], ex["annotation"][0]["points"][0]["text"][:60])

In [ ]:
# Convert each resume's annotations into clean character spans: (start, end, entity_type), dropping overlaps, blank and unlabeled spans
def to_spans(rec):
    text, spans = rec["content"], []
    for ann in rec.get("annotation", []):
        labels = ann.get("label") or []                    # some annotations carry an EMPTY label list -> skip them
        label = labels[0] if isinstance(labels, list) and labels else labels
        if not label:
            continue
        for pt in (ann.get("points") or []):
            if pt.get("start") is None or pt.get("end") is None:
                continue
            s, e = int(pt["start"]), int(pt["end"]) + 1    # Dataturks offsets are INCLUSIVE on the end, so we extend by 1
            if 0 <= s < e <= len(text) and (pt.get("text") or "").strip():
                spans.append((s, e, label))
    spans.sort(key=lambda x: (x[0], x[1]))
    clean, prev_end = [], -1                               # keep only non-overlapping spans: first annotation wins
    for s, e, l in spans:
        if s >= prev_end:
            clean.append((s, e, l)); prev_end = e
    return text, clean

parsed = [to_spans(r) for r in records]
entity_types = sorted({l for _, sp in parsed for _, _, l in sp})
print("Entity types found:", entity_types)
flat = pd.Series([l for _, sp in parsed for _, _, l in sp]).value_counts()
print("\nSpans per entity type:\n", flat)

In [ ]:
# Plot the entity-type distribution — this shows class imbalance, the reason some entities score lower than others
plt.figure(figsize=(9, 4.5))
flat.plot(kind="bar", color="#55a868")
plt.title("Entity spans per type (class imbalance visible)")
plt.ylabel("Number of spans"); plt.tight_layout()
os.makedirs("/content/artifacts/resume_ner", exist_ok=True)
plt.savefig("/content/artifacts/resume_ner/entity_distribution.png", dpi=150)
plt.show()

In [ ]:
# Split by RESUME (not by span) so no resume leaks from train into test
train_data, test_data = train_test_split(parsed, test_size=0.15, random_state=SEED)
print(f"Train resumes: {len(train_data)}  Test resumes: {len(test_data)}")

## 2. BIO tagging + tokenization
Each token gets `B-<TYPE>` (begins an entity), `I-<TYPE>` (continues one) or `O` (outside). Special tokens get `-100`, the label PyTorch ignores in the loss. This offset-mapping step is the classic **NER label-alignment pitfall** with subword tokenizers — it is the single most examinable detail of this notebook.

In [ ]:
# Tokenize with offset mapping and assign BIO labels per subtoken: tokens starting exactly at a span get B-, tokens inside get I-
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tag_set = ["O"]
for et in entity_types:
    safe = et.replace(" ", "_")                    # BIO tags must not contain spaces
    tag_set += [f"B-{safe}", f"I-{safe}"]
tag2id = {t: i for i, t in enumerate(tag_set)}
id2tag = {i: t for t, i in tag2id.items()}

def encode_resume(item):
    text, spans = item
    enc = tok(text, truncation=True, max_length=512, return_offsets_mapping=True)
    labels = []
    for s, e in enc["offset_mapping"]:
        if s == e:                                 # [CLS]/[SEP]/padding -> ignored in the loss
            labels.append(-100); continue
        lab = "O"
        for es, ee, el in spans:
            if s >= es and e <= ee:                # token fully inside the span
                lab = ("B-" if s == es else "I-") + el.replace(" ", "_")
                break
        labels.append(tag2id[lab])
    enc.pop("offset_mapping")
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}

from datasets import Dataset
ds_train = Dataset.from_list([encode_resume(x) for x in train_data])
ds_test = Dataset.from_list([encode_resume(x) for x in test_data])
print(f"Examples: {len(ds_train)} train / {len(ds_test)} test | BIO tags: {len(tag_set)}")

## 3. Fine-tune DistilBERT for token classification
Same pretrained DistilBERT as Notebook 1, but with a token-level head: every subtoken is classified into one of the BIO tags.

In [ ]:
# Build model + Trainer with per-epoch checkpoints so a Colab timeout can resume instead of restarting
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "*** NO GPU ***")
assert torch.cuda.is_available(), "NO GPU: Colab menu -> Runtime -> Change runtime type -> T4 GPU -> Save, then Runtime -> Run all"
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
model = AutoModelForTokenClassification.from_pretrained("distilbert-base-uncased", num_labels=len(tag_set), id2label=id2tag, label2id=tag2id)

from seqeval.metrics import precision_score, recall_score, f1_score
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=2)
    t, y = [], []                                  # strip -100 positions, convert ids back to tag strings for seqeval
    for tr, pr in zip(p.label_ids, preds):
        t.append([id2tag[i] for i in tr if i != -100])
        y.append([id2tag[j] for i, j in zip(tr, pr) if i != -100])
    return {"precision": precision_score(t, y), "recall": recall_score(t, y), "f1": f1_score(t, y)}

data_collator = DataCollatorForTokenClassification(tokenizer=tok)  # pads inputs AND labels (-100) per batch; the default collator leaves labels unpadded and crashes
args = TrainingArguments(
    output_dir="/content/checkpoints/resume_ner",
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=5e-5, num_train_epochs=25,   # 5 epochs = 60 steps undertrained the model (verified: F1 0.07); 25 epochs + best-checkpoint selection is the fix
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    weight_decay=0.01, warmup_ratio=0.1, fp16=torch.cuda.is_available(),  # half precision only when a GPU is actually present
    load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True,
    save_total_limit=2, logging_steps=20, report_to="none", seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_test,
                  processing_class=tok, data_collator=data_collator, compute_metrics=compute_metrics)

In [ ]:
# Train from scratch; stale checkpoints are wiped first so a re-run can never hand you an old/broken model
import shutil
if os.path.isdir(args.output_dir):
    shutil.rmtree(args.output_dir)   # resume logic caused two silently-broken artifacts - fresh training every run (~15 min, cheap insurance)
trainer.train()

In [ ]:
# Final evaluation: entity-level precision/recall/F1 (seqeval) + per-type report saved for the Model Performance page
pred_out = trainer.predict(ds_test)
m = compute_metrics(pred_out)
final_p, final_r, final_f1 = m["precision"], m["recall"], m["f1"]
print(f"Entity-level  precision: {final_p:.4f}  recall: {final_r:.4f}  F1: {final_f1:.4f}")
assert final_f1 > 0.40, (
    f"FINAL F1={final_f1:.3f} is far below the expected 0.70+ - the model in memory never trained properly. "
    "Do NOT download the zip. Fix: Runtime -> Disconnect and delete runtime -> T4 GPU -> Run all once, top to bottom.")
from seqeval.metrics import classification_report
preds = np.argmax(pred_out.predictions, axis=2)
t_seqs = [[id2tag[i] for i in tr if i != -100] for tr in pred_out.label_ids]
p_seqs = [[id2tag[j] for i, j in zip(tr, pr) if i != -100] for tr, pr in zip(pred_out.label_ids, preds)]
report = classification_report(t_seqs, p_seqs, digits=3)
print(report)
with open("/content/artifacts/resume_ner/classification_report.txt", "w") as f:
    f.write(report)

In [ ]:
# Demo: run the trained model on one held-out resume and show the entities it extracts
from transformers import pipeline as hf_pipeline
ner_pipe = hf_pipeline("token-classification", model=trainer.model, tokenizer=tok,
                       aggregation_strategy="simple", device=0 if trainer.model.device.type == "cuda" else -1)
demo_text = test_data[0][0][:1200]
found = ner_pipe(demo_text)
for ent in found[:15]:
    print(f"  {ent['entity_group']:20s}  {demo_text[ent['start']:ent['end']][:50]}")

In [ ]:
# Save model + tokenizer + tag map + metrics as JSON, zip everything for download
out = "/content/artifacts/resume_ner"
trainer.save_model(out + "/model")
tok.save_pretrained(out + "/model")
with open(out + "/tags.json", "w") as f:
    json.dump({"tags": tag_set, "id2tag": {str(k): v for k, v in id2tag.items()}, "entity_types": entity_types}, f, indent=2)
with open(out + "/metrics.json", "w") as f:
    json.dump({"task": "resume_ner", "model": "distilbert-base-uncased-token-cls",
               "n_train_resumes": len(train_data), "n_test_resumes": len(test_data),
               "precision": round(final_p, 4), "recall": round(final_r, 4), "f1": round(final_f1, 4)}, f, indent=2)
shutil.make_archive("/content/resume_ner_artifacts", "zip", out)
print("Zipped:", os.path.getsize("/content/resume_ner_artifacts.zip") // (1024 * 1024), "MB")

In [ ]:
# Trigger the browser download — move the zip into D:\resume-analyser\notebooks\artifacts\
from google.colab import files
files.download("/content/resume_ner_artifacts.zip")

## Explain this to my examiner

**What the model is.** DistilBERT (6-layer distilled BERT) fine-tuned for **token classification**: a linear layer maps every subtoken embedding to one BIO tag. I trained it for 25 epochs (lr 5e-5, batch 16, fp16) — many short epochs because 187 training resumes is a tiny dataset; the checkpoint with the best validation F1 is selected automatically on 187 annotated resumes and validated on 33 unseen resumes.

**The label-alignment pitfall (ask me about this one).** DistilBERT's WordPiece tokenizer splits words into subtokens, but entity labels exist per *span*. I solved this with `return_offsets_mapping=True`: each token's character range is compared against the annotated spans — a token starting exactly at a span's first character gets `B-TYPE`, one inside gets `I-TYPE`, and special tokens get `-100` so the loss skips them. Dataturks also uses **inclusive** character offsets, so spans must be extended by 1 — off-by-one here silently corrupts every label.

**What the metrics mean.** NER is evaluated at the *entity* level, not the token level: an entity counts as correct only if the model found the exact span with the exact type. seqeval computes entity-level **precision** (of the entities predicted, how many were right), **recall** (of the true entities, how many were found) and their harmonic-mean **F1**.

**Syllabus topics covered.** Transformers and fine-tuning (token classification head), subword tokenization, BIO labeling scheme, embeddings (contextual subtoken embeddings vs Notebook 3's trained word embeddings), precision/recall/F1 for sequence labeling.

**Honest limitations.** 220 resumes is small — F1 lands around 0.70–0.85, and rare entity types (e.g. Name) score far below frequent ones (Skills) because of class imbalance, visible in the entity-distribution plot. More annotated data is the single biggest lever.

## Troubleshooting
- **`BIO tags` count looks wrong (< 10)** → the JSON parsing cell failed; print `records[0]` and check the file is really the Dataturks JSONL format.
- **CUDA out of memory** → set `per_device_train_batch_size=8` and rerun from the model cell.
- **Session died mid-training** → `Run all` again; the train cell auto-resumes from the last epoch checkpoint in `/content/checkpoints/resume_ner`.
- **Demo shows garbage entities** → normal if you test on your own resume text far outside the training distribution; test on a held-out resume instead.